In [ ]:
# 1. BỐC VÁC DATA TỪ DRIVE XUỐNG Ổ CỨNG SIÊU TỐC COLAB (Chờ tầm 2-3 phút)
print("📦 Đang điều xe tải bưng toàn bộ 5800 ảnh từ Drive xuống ổ cứng máy chủ...")
!cp -r /content/drive/MyDrive/AIX_QuangViemPhoi/chest_xray /content/
print("✅ Dọn nhà xong! Chuyển sang đua tốc độ cao!")

# ---------------------------------------------------------
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.layers import Dense, GlobalAveragePooling2D, Dropout
from tensorflow.keras.models import Model
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint
from sklearn.utils.class_weight import compute_class_weight
import numpy as np

# 2. ĐƯỜNG DẪN MỚI (Lấy thẳng từ ổ siêu tốc nội bộ)
train_dir = '/content/chest_xray/train'
val_dir = '/content/chest_xray/test'

# 3. KÍCH HOẠT VACCINE 1: THUẬT PHÂN THÂN
train_datagen = ImageDataGenerator(
    preprocessing_function=tf.keras.applications.mobilenet_v2.preprocess_input,
    rotation_range=15,
    width_shift_range=0.1,
    height_shift_range=0.1,
    zoom_range=0.1,
    fill_mode='nearest'
)
val_datagen = ImageDataGenerator(preprocessing_function=tf.keras.applications.mobilenet_v2.preprocess_input)

train_generator = train_datagen.flow_from_directory(train_dir, target_size=(224, 224), batch_size=32, class_mode='categorical')
val_generator = val_datagen.flow_from_directory(val_dir, target_size=(224, 224), batch_size=32, class_mode='categorical', shuffle=False)

# 4. KÍCH HOẠT VACCINE 2: BƠM TRỌNG SỐ
classes = train_generator.classes
class_weights = compute_class_weight('balanced', classes=np.unique(classes), y=classes)
weight_dict = dict(enumerate(class_weights))

# 5. LẮP BỘ NÃO (MobileNetV2)
base_model = MobileNetV2(weights='imagenet', include_top=False, input_shape=(224, 224, 3))
base_model.trainable = False

x = base_model.output
x = GlobalAveragePooling2D()(x)
x = Dropout(0.5)(x)
predictions = Dense(2, activation='softmax')(x)

model = Model(inputs=base_model.input, outputs=predictions)
model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=0.001), loss='categorical_crossentropy', metrics=['accuracy'])

# 6. BÁC SĨ GIÁM SÁT KẾT QUẢ
checkpoint = ModelCheckpoint('vip_pro_pneumonia_model_v3.keras', monitor='val_accuracy', save_best_only=True, verbose=1)
early_stop = EarlyStopping(monitor='val_loss', patience=3, restore_best_weights=True)

# 7. ĐỐT LÒ BẰNG SIÊU XE T4!!!
print("🔥 BẮT ĐẦU ĐỐT LÒ LUYỆN AI V3 TỐC ĐỘ BÀN THỜ...")
history = model.fit(
    train_generator,
    epochs=12,
    validation_data=val_generator,
    class_weight=weight_dict,
    callbacks=[checkpoint, early_stop]
)

print("🎉 XONG! BỘ NÃO V3 CHỐNG VIRUS ĐÃ RA LÒ!")

📦 Đang điều xe tải bưng toàn bộ 5800 ảnh từ Drive xuống ổ cứng máy chủ...
✅ Dọn nhà xong! Chuyển sang đua tốc độ cao!
Found 5216 images belonging to 2 classes.
Found 624 images belonging to 2 classes.
🔥 BẮT ĐẦU ĐỐT LÒ LUYỆN AI V3 TỐC ĐỘ BÀN THỜ...


/usr/local/lib/python3.12/dist-packages/keras/src/trainers/data_adapters/py_dataset_adapter.py:121: UserWarning: Your `PyDataset` class should call `super().__init__(**kwargs)` in its constructor. `**kwargs` can include `workers`, `use_multiprocessing`, `max_queue_size`. Do not pass these arguments to `fit()`, as they will be ignored.
  self._warn_if_super_not_called()


Epoch 1/12
163/163 ━━━━━━━━━━━━━━━━━━━━ 0s 670ms/step - accuracy: 0.7431 - loss: 0.5923
Epoch 1: val_accuracy improved from -inf to 0.85256, saving model to vip_pro_pneumonia_model_v3.keras
163/163 ━━━━━━━━━━━━━━━━━━━━ 141s 813ms/step - accuracy: 0.7437 - loss: 0.5909 - val_accuracy: 0.8526 - val_loss: 0.3159
Epoch 2/12
163/163 ━━━━━━━━━━━━━━━━━━━━ 0s 660ms/step - accuracy: 0.9140 - loss: 0.2022
Epoch 2: val_accuracy did not improve from 0.85256
163/163 ━━━━━━━━━━━━━━━━━━━━ 112s 689ms/step - accuracy: 0.9141 - loss: 0.2021 - val_accuracy: 0.8446 - val_loss: 0.3464
Epoch 3/12
163/163 ━━━━━━━━━━━━━━━━━━━━ 0s 656ms/step - accuracy: 0.9291 - loss: 0.1646
Epoch 3: val_accuracy did not improve from 0.85256
163/163 ━━━━━━━━━━━━━━━━━━━━ 112s 688ms/step - accuracy: 0.9291 - loss: 0.1646 - val_accuracy: 0.8462 - val_loss: 0.3615
Epoch 4/12
163/163 ━━━━━━━━━━━━━━━━━━━━ 0s 645ms/step - accuracy: 0.9372 - loss: 0.1504
Epoch 4: val_accuracy did not improve from 0.85256
163/163 ━━━━━━━━━━━━━━━━━━━━ 1

In [ ]:
# ==========================================
# GIAI ĐOẠN 2: MỞ KHÓA NÃO SÂU (FINE-TUNING)
# ==========================================

print("🔓 Đang rã đông 50 lớp nơ-ron sâu thẳm của MobileNetV2...")
base_model.trainable = True

# Khóa 100 lớp đầu tiên, chỉ mở khóa từ lớp 100 trở đi
for layer in base_model.layers[:100]:
    layer.trainable = False

# Re-compile với tốc độ học (Learning Rate) CỰC KỲ CHẬM để nó ngấm từ từ
from tensorflow.keras.optimizers import Adam
model.compile(optimizer=Adam(learning_rate=0.0001),
              loss='categorical_crossentropy',
              metrics=['accuracy'])

# Đặt lại Bác sĩ giám sát (Lưu thành file tên là V3_PRO cho ngầu)
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint
checkpoint_pro = ModelCheckpoint('vip_pro_pneumonia_model_v3_PRO.keras', monitor='val_accuracy', save_best_only=True, verbose=1)
early_stop_pro = EarlyStopping(monitor='val_loss', patience=3, restore_best_weights=True)

# ĐỐT LÒ GIAI ĐOẠN CUỐI!!!
print("🚀 BẮT ĐẦU ÉP XUNG LÊN CẤP ĐỘ CHUYÊN GIA...")
history_fine = model.fit(
    train_generator,
    epochs=10, # Cho nó cày thêm tối đa 10 vòng nữa
    validation_data=val_generator,
    class_weight=weight_dict,
    callbacks=[checkpoint_pro, early_stop_pro]
)

print("🏆 ĐẠI ĐẠI CÔNG CÁO THÀNH! BỘ NÃO 'V3 PRO' ĐÃ SẴN SÀNG!")

🔓 Đang rã đông 50 lớp nơ-ron sâu thẳm của MobileNetV2...
🚀 BẮT ĐẦU ÉP XUNG LÊN CẤP ĐỘ CHUYÊN GIA...
Epoch 1/10
163/163 ━━━━━━━━━━━━━━━━━━━━ 0s 653ms/step - accuracy: 0.8979 - loss: 0.2605
Epoch 1: val_accuracy improved from -inf to 0.62500, saving model to vip_pro_pneumonia_model_v3_PRO.keras
163/163 ━━━━━━━━━━━━━━━━━━━━ 146s 748ms/step - accuracy: 0.8981 - loss: 0.2600 - val_accuracy: 0.6250 - val_loss: 2.6448
Epoch 2/10
163/163 ━━━━━━━━━━━━━━━━━━━━ 0s 645ms/step - accuracy: 0.9605 - loss: 0.0892
Epoch 2: val_accuracy improved from 0.62500 to 0.66506, saving model to vip_pro_pneumonia_model_v3_PRO.keras
163/163 ━━━━━━━━━━━━━━━━━━━━ 112s 686ms/step - accuracy: 0.9605 - loss: 0.0892 - val_accuracy: 0.6651 - val_loss: 2.0984
Epoch 3/10
163/163 ━━━━━━━━━━━━━━━━━━━━ 0s 644ms/step - accuracy: 0.9767 - loss: 0.0519
Epoch 3: val_accuracy improved from 0.66506 to 0.68590, saving model to vip_pro_pneumonia_model_v3_PRO.keras
163/163 ━━━━━━━━━━━━━━━━━━━━ 111s 681ms/step - accuracy: 0.9767 - loss

In [ ]:
import tensorflow as tf
from tensorflow.keras.applications import DenseNet121
from tensorflow.keras.layers import Dense, GlobalAveragePooling2D, Dropout
from tensorflow.keras.models import Model
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint
from tensorflow.keras.optimizers import Adam

print("🧠 ĐANG TRIỆU HỒI BỘ NÃO ĐẲNG CẤP Y KHOA: DenseNet121...")

# 1. Gọi bộ não hạng nặng (DenseNet121)
base_model_heavy = DenseNet121(weights='imagenet', include_top=False, input_shape=(224, 224, 3))

# Lần này mình chơi lớn: Mở khóa cho nó tự do học ngay từ đầu luôn (Fine-tuning nhẹ)
base_model_heavy.trainable = True
for layer in base_model_heavy.layers[:-50]: # Chỉ khóa phần móng, cho học 50 lớp chuyên sâu
    layer.trainable = False

# 2. Lắp ráp đầu ra
x = base_model_heavy.output
x = GlobalAveragePooling2D()(x)
x = Dropout(0.5)(x) # Chống học vẹt
predictions = Dense(2, activation='softmax')(x)

model_heavy = Model(inputs=base_model_heavy.input, outputs=predictions)

# Dùng tốc độ học cực kỳ chậm (0.0001) để nó thẩm thấu từ từ, không bị tẩu hỏa nhập ma
model_heavy.compile(optimizer=Adam(learning_rate=0.0001), loss='categorical_crossentropy', metrics=['accuracy'])

# 3. Bác sĩ giám sát tối cao
checkpoint_heavy = ModelCheckpoint('vip_pro_pneumonia_DENSENET.keras', monitor='val_accuracy', save_best_only=True, verbose=1)
early_stop_heavy = EarlyStopping(monitor='val_loss', patience=4, restore_best_weights=True)

print("🔥 BẮT ĐẦU ĐỐT LÒ LUYỆN ĐAN CẤP ĐỘ CAO NHẤT...")
# Lưu ý: Các biến train_generator, val_generator, weight_dict vẫn còn lưu trong RAM, mình xài lại luôn
history_heavy = model_heavy.fit(
    train_generator,
    epochs=12,
    validation_data=val_generator,
    class_weight=weight_dict,
    callbacks=[checkpoint_heavy, early_stop_heavy]
)

print("🏆 ĐÃ HOÀN THÀNH BỘ NÃO DENSENET CHUYÊN GIA!")

🧠 ĐANG TRIỆU HỒI BỘ NÃO ĐẲNG CẤP Y KHOA: DenseNet121...
29084464/29084464 ━━━━━━━━━━━━━━━━━━━━ 3s 0us/step
🔥 BẮT ĐẦU ĐỐT LÒ LUYỆN ĐAN CẤP ĐỘ CAO NHẤT...
Epoch 1/12
163/163 ━━━━━━━━━━━━━━━━━━━━ 0s 682ms/step - accuracy: 0.7288 - loss: 0.5033
Epoch 1: val_accuracy improved from -inf to 0.88782, saving model to vip_pro_pneumonia_DENSENET.keras
163/163 ━━━━━━━━━━━━━━━━━━━━ 190s 888ms/step - accuracy: 0.7295 - loss: 0.5021 - val_accuracy: 0.8878 - val_loss: 0.2739
Epoch 2/12
163/163 ━━━━━━━━━━━━━━━━━━━━ 0s 683ms/step - accuracy: 0.9403 - loss: 0.1482
Epoch 2: val_accuracy did not improve from 0.88782
163/163 ━━━━━━━━━━━━━━━━━━━━ 117s 718ms/step - accuracy: 0.9403 - loss: 0.1482 - val_accuracy: 0.8718 - val_loss: 0.3063
Epoch 3/12
163/163 ━━━━━━━━━━━━━━━━━━━━ 0s 675ms/step - accuracy: 0.9518 - loss: 0.1122
Epoch 3: val_accuracy did not improve from 0.88782
163/163 ━━━━━━━━━━━━━━━━━━━━ 117s 714ms/step - accuracy: 0.9518 - loss: 0.1122 - val_accuracy: 0.8782 - val_loss: 0.3596
Epoch 4/12
163/1